In [1]:
import os, glob, time, json
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from transformers import AutoModel

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
BASE = os.path.dirname(hits[0])
WORK = '/kaggle/working'
print('BASE =', BASE)
print('cuda available:', torch.cuda.is_available())

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
uids = ['train/' + f for f in train.filename] + ['test/' + f for f in test.filename]
print(len(uids), 'files')

device = 'cuda'
model = AutoModel.from_pretrained('microsoft/wavlm-large').to(device).eval()
print('model loaded | hidden states per file (embedding layer + transformer layers):',
      model.config.num_hidden_layers + 1)

BASE = /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
cuda available: True
985 files


model loaded | hidden states per file (embedding layer + transformer layers): 25


In [2]:
USE_FP16 = True   # set to False if the NaN check below prints True

def load_wav(uid):
    y, sr = sf.read(f'{BASE}/{uid}', dtype='float32')
    if y.ndim > 1:
        y = y.mean(axis=1)
    assert sr == 16000, f'unexpected sample rate {sr} for {uid}'
    return (y - y.mean()) / (y.std() + 1e-7)     # level normalisation

@torch.inference_mode()
def embed(uid):
    x = torch.from_numpy(load_wav(uid))[None].to(device)
    with torch.autocast('cuda', dtype=torch.float16, enabled=USE_FP16):
        hs = model(x, output_hidden_states=True).hidden_states
    H = torch.stack([h[0].float() for h in hs], 0)      # (layers, frames, 1024)
    return H.mean(1).cpu().numpy(), H.std(1).cpu().numpy()

t0 = time.time()
for u in uids[:3]:
    m, s = embed(u)
print('shapes:', m.shape, s.shape, '| seconds per file (first call includes warm-up):',
      round((time.time() - t0) / 3, 2))
print('NaN present:', bool(np.isnan(m).any() or np.isnan(s).any()))

shapes: (25, 1024) (25, 1024) | seconds per file (first call includes warm-up): 3.57
NaN present: False


In [3]:
N, L, D = len(uids), model.config.num_hidden_layers + 1, model.config.hidden_size
P_MEAN, P_STD, P_N = f'{WORK}/_part_mean.npy', f'{WORK}/_part_std.npy', f'{WORK}/_part_n.json'

if os.path.exists(P_N):
    feats_mean, feats_std = np.load(P_MEAN), np.load(P_STD)
    start = json.load(open(P_N))['n_done']
    print('resuming from', start)
else:
    feats_mean = np.zeros((N, L, D), dtype=np.float16)
    feats_std = np.zeros((N, L, D), dtype=np.float16)
    start = 0

t0 = time.time()
for i in range(start, N):
    m, s = embed(uids[i])
    feats_mean[i], feats_std[i] = m, s
    if (i + 1) % 100 == 0 or i + 1 == N:
        np.save(P_MEAN, feats_mean); np.save(P_STD, feats_std)
        json.dump({'n_done': i + 1}, open(P_N, 'w'))
        el = time.time() - t0
        print(f'{i + 1}/{N} | {el / 60:.1f} min | est remaining {(N - i - 1) * el / (i + 1 - start) / 60:.1f} min')

print('NaN or inf present:', bool(np.isnan(feats_mean).any() or np.isinf(feats_mean).any()
                                  or np.isnan(feats_std).any() or np.isinf(feats_std).any()))
print('all-zero rows:', int((np.abs(feats_mean).sum(axis=(1, 2)) == 0).sum()))

np.save(f'{WORK}/wavlm_large_mean.npy', feats_mean)
np.save(f'{WORK}/wavlm_large_std.npy', feats_std)
pd.Series(uids).to_csv(f'{WORK}/wavlm_uids.csv', index=False, header=['uid'])
for p in (P_MEAN, P_STD, P_N):
    os.remove(p)
print('saved', feats_mean.shape, '| files in working dir:', sorted(os.listdir(WORK)))

100/985 | 2.1 min | est remaining 18.3 min
200/985 | 4.2 min | est remaining 16.5 min
300/985 | 6.3 min | est remaining 14.3 min
400/985 | 8.4 min | est remaining 12.2 min
500/985 | 10.4 min | est remaining 10.1 min
600/985 | 12.5 min | est remaining 8.0 min
700/985 | 14.7 min | est remaining 6.0 min
800/985 | 16.6 min | est remaining 3.8 min
900/985 | 18.2 min | est remaining 1.7 min
985/985 | 19.6 min | est remaining 0.0 min
NaN or inf present: False
all-zero rows: 0
saved (985, 25, 1024) | files in working dir: ['__notebook__.ipynb', 'wavlm_large_mean.npy', 'wavlm_large_std.npy', 'wavlm_uids.csv']


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:53: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
